# Notebook 3 — Building a RAG-Based Q&A Flow

### Build an AI Business Copilot 🏔️ · Session 1

So far:
- **Notebook 1:** an LLM doesn't know your business — you have to **ground** it with facts.
- **Notebook 2:** we built a **retriever** that automatically *finds* the right document
  chunk for any question.

Now we connect the two into a complete **RAG (Retrieval-Augmented Generation)** flow:

> **Retrieve** the relevant chunks → **hand them to Claude** → get a grounded, cited answer.

By the end you'll have a working **document Q&A copilot** and you'll be able to:

1. Assemble a **grounded prompt** from retrieved chunks.
2. Get Claude to answer *only* from those chunks — and **cite** its sources.
3. Make the copilot **say "I don't know"** instead of guessing.
4. Wrap it all into one reusable `answer_question()` function.

> 💸 This notebook makes a few short Claude calls (Haiku 4.5) — a fraction of a cent total.

## 1. Setup

Install the libraries, fetch the workshop files, and enter your API key (we call Claude in
this notebook). Same setup pattern as Notebook 2 — run the three cells below.

In [ ]:
%pip install -q anthropic sentence-transformers faiss-cpu pandas numpy

In [ ]:
# --- Make the workshop files available (Colab, Databricks, or local) ---
import sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/allistaircota/trailhead-copilot"   # <-- ✏️ set to the real repo URL

def find_repo_root():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if all((base / d).is_dir() for d in ("data", "documents", "src")):
            return base
    cloned = Path.cwd() / "trailhead-copilot"
    return cloned if (cloned / "src").is_dir() else None

root = find_repo_root()
if root is None:
    print("Cloning the workshop repo...")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
    root = find_repo_root()
if root is None:
    raise RuntimeError("Workshop files not found — set REPO_URL to the real repository URL.")

sys.path.insert(0, str(root / "src"))
print("Using workshop files at:", root)

In [ ]:
import os, getpass
if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Paste your Anthropic API key: ")
print("API key is set. ✅")

## 2. Configuration

In [ ]:
MODEL = "claude-haiku-4-5"   # cheap and capable for grounded Q&A
MAX_TOKENS = 400             # room for a full answer with a citation
TOP_K = 4                    # how many chunks to retrieve per question

## 3. Rebuild the retriever (from Notebook 2)

We packaged Notebook 2's work — chunking, embedding, and search — into `trailhead.py`, so
we can set the retriever up in three lines instead of retyping it. (`build_index`
downloads the embedding model on first run; give it a moment.)

In [ ]:
import trailhead as th

chunks = th.build_chunks()                 # load + chunk every document
index, embedder = th.build_index(chunks)   # embed the chunks and build the FAISS index

def retrieve(query, k=TOP_K):
    # Returns the top-k chunks, each with a similarity 'score'.
    return th.search(query, index, chunks, embedder, top_k=k)

# Sanity check: the retriever still works.
for hit in retrieve("return window for boots", k=2):
    print(f"[{hit['score']:.2f}] {hit['source']}")

## 4. The anatomy of a grounded (RAG) prompt

This is the heart of the copilot. We build a prompt with three parts:

1. **The retrieved context** — the chunks the retriever found, each labeled with its
   source document.
2. **The customer's question.**
3. **The rules** (in the system prompt) — the two habits from Notebook 1, plus a request to
   cite sources:
   - *Answer using only the provided excerpts.*
   - *If it's not there, say you don't know* (don't guess!).
   - *Cite which document you used.*

Let's write the helpers that assemble this, and **print the prompt** so you can see exactly
what Claude receives before we send it.

In [ ]:
SYSTEM = (
    "You are a customer support assistant for Trailhead Supply Co., an outdoor gear "
    "retailer. Answer the customer's question using ONLY the policy excerpts provided. "
    "If the answer is not in the excerpts, say you don't have that information and "
    "suggest emailing support@trailheadsupply.example. Keep answers short and friendly, "
    "and cite the document you used like [source: return_policy.md]."
)

def build_context(hits):
    # Join the retrieved chunks into one labeled context block.
    blocks = []
    for h in hits:
        blocks.append("[source: " + h["source"] + "]\n" + h["text"])
    return "\n\n".join(blocks)

def build_prompt(query, hits):
    return "POLICY EXCERPTS:\n" + build_context(hits) + "\n\nCUSTOMER QUESTION: " + query

# See what a real grounded prompt looks like:
example_hits = retrieve("Can I return a final sale item?")
print(build_prompt("Can I return a final sale item?", example_hits))

## 5. Your first grounded answer

Now we send that prompt to Claude and read the answer. We'll print the answer *and* the
sources the retriever pulled, so you can check the answer against the evidence.

In [ ]:
query = "Can I return a final sale item?"
hits = retrieve(query)

answer = th.ask_claude(build_prompt(query, hits), system=SYSTEM, model=MODEL, max_tokens=MAX_TOKENS)

print("QUESTION:", query, "\n")
print("ANSWER:\n", answer, "\n")
print("Retrieved from:", ", ".join(sorted({h["source"] for h in hits})))

Compare this with Notebook 1, where Claude *guessed* about final sale returns. Now it
answers correctly — *"Final Sale items are not returnable"* — and **cites the document it
used**. Same model, no training. The only difference is that we gave it the right facts.

## 6. Trust, but verify — citations

Notice the `[source: ...]` tag in the answer. Citations matter in business: they let a
human (or the customer) check where an answer came from. Because we also printed the
retrieved chunks, you can confirm the answer is genuinely **grounded** in them and not
invented. This verifiability is a big part of why RAG is trusted for real support work.

## 7. The "I don't know" guardrail

A good copilot knows its limits. If the answer isn't in Trailhead's documents, it should
**say so** rather than make something up. Let's ask about something our documents don't
cover — gift cards — and watch the guardrail work.

In [ ]:
query = "Do you sell gift cards?"
hits = retrieve(query)

answer = th.ask_claude(build_prompt(query, hits), system=SYSTEM, model=MODEL, max_tokens=MAX_TOKENS)

print("ANSWER:\n", answer, "\n")
print("Best retrieved chunk score:", round(hits[0]["score"], 2), "(low = weak match)")

Instead of inventing a gift-card policy, the copilot admits it doesn't have that
information and points the customer to support. That single instruction — *"if it's not in
the excerpts, say you don't know"* — is one of the most valuable guardrails in a business
copilot.

## 8. Package it into `answer_question()`

Let's bundle retrieve → build prompt → ask Claude into one clean function. This is the
document-Q&A copilot — and we'll reuse it in Notebook 5 when we route questions.

In [ ]:
def answer_question(query, k=TOP_K):
    hits = retrieve(query, k)
    answer = th.ask_claude(build_prompt(query, hits), system=SYSTEM, model=MODEL, max_tokens=MAX_TOKENS)
    return {
        "answer": answer,
        "sources": sorted({h["source"] for h in hits}),
    }

result = answer_question("How do I take care of my down sleeping bag?")
print(result["answer"])
print("\nSources:", ", ".join(result["sources"]))

> ✏️ **Try it.** Put your own questions in the list below and run it. Try a policy question,
> a gear-care question, and something the documents *don't* cover — watch the copilot answer
> the first two and decline the third.

In [ ]:
for q in [
    "What does the warranty cover on a tent?",
    "How long does standard shipping take?",
    "Do you offer a student discount?",
]:
    print("Q:", q)
    print("A:", answer_question(q)["answer"])
    print("-" * 70)

## 9. What we built, and what's next

You now have a complete **document Q&A copilot**: it retrieves the right policy text and
answers from it, with citations and a "don't guess" guardrail. That's the full RAG loop —
**Retrieve + Generate** — and the whole of the diagram's left branch from Notebook 1.

But customers also ask questions that **aren't** in any document — *"Where is my order?"*,
*"Is this tent in stock?"* Those answers live in Trailhead's **databases**, not its policies.

**Notebook 4** teaches the copilot to look up structured business data (orders, inventory,
shipments) using **tool use**, so it can answer account-specific questions with real
records instead of guessing.

## 10. Recap

- **RAG = Retrieve + Generate.** Find the relevant chunks, then have Claude answer from
  them.
- A **grounded prompt** = retrieved context + the question + rules (use only this, cite
  sources, say "I don't know" when unsure).
- **Citations** make answers verifiable; the **"I don't know" guardrail** prevents
  confident wrong answers.
- `answer_question()` is your reusable document-Q&A copilot.

## 11. Exercises

**Guided**
1. Ask `answer_question()` a **warranty** question and a **shipping** question. Check that
   the `sources` it reports make sense for each.
2. Edit the `SYSTEM` prompt to make the assistant answer in **exactly one sentence**.
   Re-run a couple of questions. What changed?

**Challenge**
3. Lower `TOP_K` to `1` and ask a question whose answer spans two paragraphs (e.g., returns
   *and* the member extension). Does one chunk give Claude enough to answer well? Raise
   `TOP_K` back up and compare.
4. Add a "confidence" note: modify `answer_question()` to also return the **top retrieval
   score**, and print a warning when it's below, say, `0.2` ("weak match — answer may be
   unreliable"). This is a simple, practical guardrail you could ship.

## 12. Learn more

- **Anthropic — Prompt engineering: be clear & direct:** <https://docs.claude.com/en/docs/build-with-claude/prompt-engineering/be-clear-and-direct>
- **Anthropic — Reduce hallucinations:** <https://docs.claude.com/en/docs/test-and-evaluate/strengthen-guardrails/reduce-hallucinations>
- **Anthropic — Messages API reference:** <https://docs.claude.com/en/api/messages>
- **IBM — Retrieval-Augmented Generation (RAG):** <https://www.ibm.com/think/topics/retrieval-augmented-generation>